# MMRL run 2: ScienceQA with multiple-choice option scoring
Reuses the ogbn-arxiv and ROCOv2 results and data from run 1 (`notebook2b5b23c132`, attached as input). Re-runs ScienceQA (main grid + ablations) with each answer option scored against the document embedding, then rebuilds every table, figure and statistic. Settings: **GPU T4 x2, Internet On**.

In [ ]:
import base64, io, tarfile, os, glob, shutil, json
CODE = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode='r:gz').extractall('/kaggle/working')
prev = [d for d in glob.glob('/kaggle/input/**/results', recursive=True) if os.path.isdir(d)]
print('previous results folders:', prev)
assert prev, 'attach the output of notebook2b5b23c132 (run 1) as input'
P = os.path.dirname(prev[0])
os.makedirs('/kaggle/working/results', exist_ok=True)
kept = 0
for f in glob.glob(P + '/results/*'):
    b = os.path.basename(f)
    if b.startswith('scienceqa__') or b == 'cost_scienceqa' or b == 'stats.json':
        continue
    dst = '/kaggle/working/results/' + b
    (shutil.copytree if os.path.isdir(f) else shutil.copy)(f, dst); kept += 1
for ds in ('ogbn_arxiv', 'rocov2'):
    shutil.copytree(f'{P}/data/{ds}', f'/kaggle/working/data/{ds}', symlinks=False, dirs_exist_ok=True)
print('kept', kept, 'result files from run 1; data:', os.listdir('/kaggle/working/data'))

In [ ]:
!pip install -q huggingface_hub 2>&1 | tail -1
import torch, transformers, sys
print(sys.version.split()[0], 'torch', torch.__version__, 'transformers', transformers.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
%cd /kaggle/working/mmrl
from huggingface_hub import snapshot_download
snapshot_download('derek-thomas/ScienceQA', repo_type='dataset', local_dir='/tmp/sqa', allow_patterns=['data/*'])
!python tools/prepare_scienceqa.py --src /tmp/sqa/data --out /kaggle/working/data/scienceqa --max_text_len 256
!python tools/embed_entities.py /kaggle/working/data/scienceqa
!python tools/check_dataset.py /kaggle/working/data/scienceqa
recs=[json.loads(l) for l in open('/kaggle/working/data/scienceqa/test.jsonl')]
print('ScienceQA test chance accuracy (mean 1/n_choices): %.2f%%' % (100*sum(1/r['n_choices'] for r in recs)/len(recs)))

In [ ]:
import subprocess, time
env = dict(os.environ, AMP='fp16', BS='16', ACCUM='2', EPOCHS='4', PATIENCE='2', SEEDS='0 1 2', WORKERS='2')
def job(gpu, stage, log):
    return subprocess.Popen(f'CUDA_VISIBLE_DEVICES={gpu} STAGE={stage} bash kaggle/session.sh > {log} 2>&1', shell=True, env=env)
jobs = [job(0, 'sqa', '/kaggle/working/log_gpu0.txt'), job(1 if torch.cuda.device_count() > 1 else 0, 'ablation', '/kaggle/working/log_gpu1.txt')]
t0 = time.time()
while any(j.poll() is None for j in jobs):
    time.sleep(600)
    n = len(glob.glob('/kaggle/working/results/scienceqa__*.json'))
    print(f'{(time.time()-t0)/3600:.1f} h, {n} ScienceQA runs finished', flush=True)
    if (time.time() - t0) / 3600 > 10.5:
        subprocess.run('pkill -f train.py', shell=True); break
print('grid done in', round((time.time()-t0)/3600, 2), 'h')
!grep -h -E "FAILED|Error" /kaggle/working/log_gpu*.txt | tail -20

In [ ]:
# cost for the new ScienceQA configuration, then the full report printed in the log
for m in ['mmrl', 'cma', 'mmbert', 'late', 'early']:
    subprocess.run(f'CUDA_VISIBLE_DEVICES=0 python profile_cost.py --data /kaggle/working/data/scienceqa --method {m} --amp fp16 --out /kaggle/working/results/cost_scienceqa > /dev/null 2>&1', shell=True)
subprocess.run('CUDA_VISIBLE_DEVICES=0 python profile_cost.py --data /kaggle/working/data/scienceqa --method unimodal --unimodal text --amp fp16 --out /kaggle/working/results/cost_scienceqa > /dev/null 2>&1', shell=True)
!STAGE=report bash kaggle/session.sh > /kaggle/working/report_log.txt 2>&1; tail -3 /kaggle/working/report_log.txt
print('=====TABLES_MD_BEGIN====='); print(open('/kaggle/working/paper_out/tables.md').read()); print('=====TABLES_MD_END=====')
print('=====STATS_BEGIN====='); print(open('/kaggle/working/paper_out/stats.json').read()); print('=====STATS_END=====')
S = json.load(open('/kaggle/working/paper_out/summary.json'))
for k in ('table5', 'convergence', 'confusion', 'auc', 'robustness', 'align_quartiles', 'cost_vs_cma', 'unimodal_choice', 'gain_over_strongest'):
    print(f'=====SUMMARY_{k}====='); print(json.dumps(S.get(k), indent=0)[:6000])
!cp /kaggle/working/log_gpu*.txt /kaggle/working/paper_out/; cd /kaggle/working && zip -qr paper_out.zip paper_out && zip -qr results.zip results && ls -la /kaggle/working